# EDA CHALLENGE LEADERBOARDER

In [ ]:
import sys
import numpy as np
import pandas as pd
import json

from pathlib import Path

sys.path.append("..")
from src.utils.utils import *

# Cargamos el dataset
df = load_data_csv(r'..\data\raw\train.csv')

# Cargamos el dataset Summary
df_summary = load_json(r'..\data\dataset_summary.json')

## ANALISIS VISUAL

### CONCLUSIÓN

Para comenzar el análisis exploratorio de los datos, vamos a realizar un primer análisis que nos permita conocer el estado de los datos y entender qué representa cada una de las variables, con el objetivo de poder realizar correctamente el preprocesamiento antes de entrenar el modelo. Para ello, utilizaremos la herramienta *Data Wrangler*, que nos permite observar la estructura del dataset, los tipos de datos y los distintos valores que puede tomar cada columna, entre otras cosas.

Tras este primer análisis, hemos podido entender que cada registro asociado a un *timestamp* corresponde a una trayectoria realizada por un sujeto. Cada una de estas trayectorias está formada por 200 muestras, registradas a una frecuencia de 50 Hz, por lo que cada trayectoria tiene una duración de 4 segundos.

Por otro lado, al representar los datos en el espacio cartesiano, hemos observado que los sensores presentan algunos posibles *outliers*, ya que aparecen puntos que se alejan considerablemente del resto de la trayectoria. Para corregir estos valores, no podemos eliminarlos directamente, puesto que pueden aparecer en un eje espacial concreto y eliminar una muestra supondría acortar la trayectoria. Esto podría provocar que las distintas trayectorias tuviesen un número diferente de muestras, algo que perjudicaría el entrenamiento del modelo.

Por este motivo, dado que estamos trabajando con series temporales de movimiento, vamos a representar las trayectorias teniendo en cuenta también el tiempo. De esta forma, podremos observar si existe una dispersión considerable entre los puntos y estudiar cómo filtrarla. Para ello, debemos tener en cuenta que las muestras de distintas trayectorias no tienen por qué corresponder exactamente al mismo instante, aunque sí pueden pertenecer a momentos cercanos dentro de un cierto margen de tiempo.

Como se puede observar en las gráficas de dispersión por sensor y sesión, las medidas registradas por cada sensor presentan diferentes niveles de dispersión. Es decir, algunos sensores muestran mayores diferencias en la posición de los puntos que otros. Por este motivo, para realizar el análisis temporal y corregir los posibles *outliers*, vamos a tratar cada uno de los sensores por separado.

### Analsiis Espacial

#### ESPACIO CARTESIANO POR SUJETO

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Classes = df_summary["activity_classes"]
subjects = list(df_summary["train_subject_distribution"].keys())
sensores = list(df_summary["channel_names"])

# Identificar articulaciones
articulaciones = []

for s in sensores:
    base = s[:-2] if s.endswith(("_x", "_y", "_z")) else s

    if base not in articulaciones:
        articulaciones.append(base)

# Colores por sujeto
cmap = plt.get_cmap("tab20", max(len(subjects), 1))

subject_colors = {
    subj: cmap(i) for i, subj in enumerate(subjects)
}

# Número de instantes temporales
n_timesteps = 200

for type_class in Classes:

    df_class = df[df["label"] == type_class].copy()

    if df_class.empty:
        continue

    for start in range(0, len(articulaciones), 4):

        bloque = articulaciones[start:start + 4]

        fig, axes = plt.subplots(
            1, 4,
            figsize=(24, 6),
            subplot_kw={"projection": "3d"}
        )

        fig.suptitle(
            f"Movimiento: {type_class} | Todos los sujetos "
            f"({start // 4 + 1})",
            fontsize=14
        )

        for idx, ax in enumerate(axes):

            ax.grid(True)

            if idx >= len(bloque):
                ax.set_visible(False)
                continue

            sensor = bloque[idx]

            cols_x = [
                f"{sensor}_x_t{t}"
                for t in range(n_timesteps)
                if f"{sensor}_x_t{t}" in df_class.columns
            ]

            cols_y = [
                f"{sensor}_y_t{t}"
                for t in range(n_timesteps)
                if f"{sensor}_y_t{t}" in df_class.columns
            ]

            cols_z = [
                f"{sensor}_z_t{t}"
                for t in range(n_timesteps)
                if f"{sensor}_z_t{t}" in df_class.columns
            ]

            if not (cols_x and cols_y and cols_z):
                continue

            # Representar cada sujeto
            for subject in subjects:

                df_subj = df_class[
                    df_class["subject_id"].astype(str) == str(subject)
                ]

                if df_subj.empty:
                    continue

                arr_x = df_subj[cols_x].to_numpy(dtype=float)
                arr_y = df_subj[cols_y].to_numpy(dtype=float)
                arr_z = df_subj[cols_z].to_numpy(dtype=float)

                # Cada trayectoria se representa como puntos espaciales
                for i in range(len(df_subj)):

                    ax.scatter(
                        arr_x[i],
                        arr_y[i],
                        arr_z[i],
                        color=subject_colors[subject],
                        s=5,
                        alpha=0.35,
                        label=str(subject) if i == 0 else None
                    )

            ax.set_title(sensor)
            ax.set_xlabel("X")
            ax.set_ylabel("Y")
            ax.set_zlabel("Z")

        # Leyenda ordenada por sujeto
        handles_dict = {}

        for ax in axes:
            if not ax.get_visible():
                continue

            handles, labels = ax.get_legend_handles_labels()

            for handle, label in zip(handles, labels):
                if label not in handles_dict:
                    handles_dict[label] = handle

        labels_ordered = [
            str(subject)
            for subject in subjects
            if str(subject) in handles_dict
        ]

        handles_ordered = [
            handles_dict[label]
            for label in labels_ordered
        ]

        if handles_ordered:
            fig.legend(
                handles_ordered,
                labels_ordered,
                loc="upper right",
                ncol=2,
                title="Sujetos"
            )

        plt.tight_layout(rect=[0, 0, 0.92, 0.94])
        plt.show()


####  ESPACIO CARTESIANO - POR SENSOR Y SESIÓN

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Classes = df_summary["activity_classes"]

# SELECCIÓN DE SUJETOS
subjects = list(df_summary["train_subject_distribution"].keys())[7:8]
subjects = [str(s) for s in subjects]

# SESIONES Y DISPOSITIVOS
sessions = ["S1", "S2"]
device_types = ["Commercial", "Custom"]

session_colors = {
    "S1": "red",
    "S2": "blue"
}

sensores = list(df_summary["channel_names"])

articulaciones = []
for s in sensores:
    base = s[:-2] if s.endswith(("_x", "_y", "_z")) else s
    if base not in articulaciones:
        articulaciones.append(base)


def ajustar_tono(color, factor, hacia_blanco=True):

    rgba = np.array(mcolors.to_rgba(color), dtype=float)

    objetivo = (
        np.array([1.0, 1.0, 1.0, rgba[3]])
        if hacia_blanco
        else np.array([0.0, 0.0, 0.0, rgba[3]])
    )

    resultado = rgba * (1.0 - factor) + objetivo * factor
    resultado[:3] = np.clip(resultado[:3], 0.0, 1.0)

    return tuple(resultado)


# COLORES POR SESIÓN Y TIPO DE DISPOSITIVO
colores = {}

for session in sessions:

    base_color = session_colors[session]

    colores[(session, "Commercial")] = ajustar_tono(
        base_color, 0.45, hacia_blanco=True
    )

    colores[(session, "Custom")] = ajustar_tono(
        base_color, 0.35, hacia_blanco=False
    )


df = df.copy()
df["subject_id"] = df["subject_id"].astype(str)

for type_class in Classes:

    df_class = df[df["label"] == type_class].copy()

    if df_class.empty:
        continue

    for start in range(0, len(articulaciones), 4):

        bloque = articulaciones[start:start + 4]

        fig, axes = plt.subplots(
            1, 4,
            figsize=(24, 6),
            subplot_kw={"projection": "3d"}
        )

        fig.suptitle(
            f"Movimiento: {type_class} | "
            f"S1 (rojo) vs S2 (azul) | "
            f"Commercial (claro) vs Custom (oscuro) "
            f"({start // 4 + 1})",
            fontsize=14
        )

        for idx, ax in enumerate(axes):

            ax.grid(True)

            if idx >= len(bloque):
                ax.set_visible(False)
                continue

            sensor = bloque[idx]

            # Instantes temporales disponibles en los tres ejes
            times = [
                t for t in range(200)
                if all(
                    f"{sensor}_{axis}_t{t}" in df_class.columns
                    for axis in ["x", "y", "z"]
                )
            ]

            if not times:
                continue

            cols_x = [f"{sensor}_x_t{t}" for t in times]
            cols_y = [f"{sensor}_y_t{t}" for t in times]
            cols_z = [f"{sensor}_z_t{t}" for t in times]

            for subject in subjects:

                df_subj = df_class[
                    df_class["subject_id"] == subject
                ]

                if df_subj.empty:
                    continue

                for session in sessions:

                    df_session = df_subj[
                        df_subj["session"] == session
                    ]

                    if df_session.empty:
                        continue

                    for dev_type in device_types:

                        df_dev = df_session[
                            df_session["device_type"] == dev_type
                        ]

                        if df_dev.empty:
                            continue

                        arr_x = df_dev[cols_x].to_numpy(dtype=float)
                        arr_y = df_dev[cols_y].to_numpy(dtype=float)
                        arr_z = df_dev[cols_z].to_numpy(dtype=float)

                        color = colores[(session, dev_type)]

                        # Cada instante temporal se representa como un punto
                        for i in range(len(df_dev)):

                            ax.scatter(
                                arr_x[i],
                                arr_y[i],
                                arr_z[i],
                                color=color,
                                s=5,
                                alpha=0.35,
                                depthshade=False,
                                label=(
                                    f"{subject} - {session} ({dev_type})"
                                    if i == 0 else None
                                )
                            )

            ax.set_title(sensor)
            ax.set_xlabel("x")
            ax.set_ylabel("y")
            ax.set_zlabel("z")

        # LEYENDA ORDENADA:
        # S1 Commercial, S1 Custom, S2 Commercial, S2 Custom
        handles_dict = {}

        for ax in axes:
            if not ax.get_visible():
                continue

            handles, labels = ax.get_legend_handles_labels()

            for handle, label in zip(handles, labels):
                if label not in handles_dict:
                    handles_dict[label] = handle

        labels_ordered = [
            f"{subject} - {session} ({dev_type})"
            for subject in subjects
            for session in sessions
            for dev_type in device_types
            if f"{subject} - {session} ({dev_type})" in handles_dict
        ]

        handles_ordered = [
            handles_dict[label] for label in labels_ordered
        ]

        if handles_ordered:
            fig.legend(
                handles_ordered,
                labels_ordered,
                loc="upper right",
                ncol=2,
                title="Sujeto - Sesión (Device Type)"
            )

        plt.tight_layout(rect=[0, 0, 0.90, 0.94])
        plt.show()


####  ESPACIO CARTESIANO DISTRUBICIÓN (CAJA Y BIGOTE)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Classes = df_summary["activity_classes"]
subjects = list(df_summary["train_subject_distribution"].keys())
sensores = list(df_summary["channel_names"])

# Obtener articulaciones únicas
articulaciones = []
for s in sensores:
    base = s[:-2] if s.endswith(("_x", "_y", "_z")) else s
    if base not in articulaciones:
        articulaciones.append(base)

# Un color por sujeto
cmap = plt.get_cmap("tab20", max(len(subjects), 1))
subject_colors = {
    subj: cmap(i) for i, subj in enumerate(subjects)
}

# Una figura por movimiento
for type_class in Classes:

    df_class = df[df["label"] == type_class].copy()

    if df_class.empty:
        continue

    n_articulaciones = len(articulaciones)

    fig, axes = plt.subplots(
        3,
        n_articulaciones,
        figsize=(max(5 * n_articulaciones, 14), 13),
        squeeze=False
    )

    fig.suptitle(
        f"Movimiento: {type_class} | Distribución espacial",
        fontsize=17,
        y=1.02
    )

    # Recorrer articulaciones
    for col, sensor in enumerate(articulaciones):

        # Recorrer ejes X, Y, Z
        for row, eje in enumerate(["x", "y", "z"]):

            ax = axes[row, col]

            datos_violin = []
            posiciones = []
            colores_violin = []

            # Obtener datos de cada sujeto
            for pos, subject in enumerate(subjects):

                df_subj = df_class[
                    df_class["subject_id"].astype(str) == str(subject)
                ]

                if df_subj.empty:
                    continue

                cols = [
                    f"{sensor}_{eje}_t{t}"
                    for t in range(200)
                    if f"{sensor}_{eje}_t{t}" in df_subj.columns
                ]

                if not cols:
                    continue

                valores = df_subj[cols].to_numpy(dtype=float).flatten()
                valores = valores[np.isfinite(valores)]

                if len(valores) == 0:
                    continue

                datos_violin.append(valores)
                posiciones.append(pos)
                colores_violin.append(subject_colors[subject])

            # Dibujar violines
            if datos_violin:

                violines = ax.violinplot(
                    datos_violin,
                    positions=posiciones,
                    widths=0.8,
                    showmeans=False,
                    showmedians=True,
                    showextrema=True
                )

                # Aplicar un color diferente por sujeto
                for violin, color in zip(
                    violines["bodies"], colores_violin
                ):
                    violin.set_facecolor(color)
                    violin.set_edgecolor("black")
                    violin.set_alpha(0.65)

                # Personalizar mediana y extremos
                for elemento in ["cmedians", "cmins", "cmaxes", "cbars"]:
                    violines[elemento].set_color("black")
                    violines[elemento].set_linewidth(1)

            # Personalización
            ax.set_title(sensor, fontsize=11)
            ax.set_ylabel(f"Eje {eje.upper()} - Valor")
            ax.grid(True, axis="y", alpha=0.3)

            ax.set_xticks(range(len(subjects)))
            ax.set_xticklabels(
                [str(s) for s in subjects],
                rotation=45,
                ha="right"
            )

            ax.set_xlim(-0.5, len(subjects) - 0.5)

            if row == 2:
                ax.set_xlabel("Sujetos")

    plt.tight_layout()
    plt.show()


#### Rrepresentación de trayectorias por sample

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ==========================================
# 1. Seleccionar el caso de estudio
# ==========================================

subject         = "P1"
device          = "Commercial"  # Commercial o Custom
session         = "S1"         # S1 o S2
movement        = "Drive"
articulacion    = "wrist_R"

# Número máximo de timestamps que quieres representar
max_timestamps = 5

# ==========================================
# 2. Filtrar los datos
# ==========================================

df_plot = df[
    (df["subject_id"] == subject) &
    (df["device_type"] == device) &
    (df["session"] == session) &
    (df["label"] == movement)
].copy()

# Ordenar cronológicamente
df_plot = df_plot.sort_values("timestamp").reset_index(drop=True)

if df_plot.empty:
    raise ValueError("No hay datos para el caso seleccionado.")

# Limitar el número de timestamps
df_plot = df_plot.iloc[:max_timestamps]

# ==========================================
# 3. Seleccionar coordenadas X, Y, Z
# ==========================================

cols_x = [f"{articulacion}_x_t{i}" for i in range(200)]
cols_y = [f"{articulacion}_y_t{i}" for i in range(200)]
cols_z = [f"{articulacion}_z_t{i}" for i in range(200)]

columnas = cols_x + cols_y + cols_z

faltantes = [c for c in columnas if c not in df_plot.columns]

if faltantes:
    raise ValueError(f"Columnas no encontradas: {faltantes[:10]}")

# ==========================================
# 4. Crear colores para cada timestamp
# ==========================================

n_timestamps = len(df_plot)
cmap = plt.colormaps["turbo"].resampled(n_timestamps)

# ==========================================
# 5. Representar cada trayectoria
# ==========================================

fig = plt.figure(figsize=(14, 11))
ax = fig.add_subplot(111, projection="3d")

for i, (_, row) in enumerate(df_plot.iterrows()):

    x = row[cols_x].to_numpy(dtype=float)
    y = row[cols_y].to_numpy(dtype=float)
    z = row[cols_z].to_numpy(dtype=float)

    validos = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)

    x = x[validos]
    y = y[validos]
    z = z[validos]

    if len(x) == 0:
        continue

    color = cmap(i)

    # Línea de la trayectoria
    ax.plot(
        x, y, z,
        color=color,
        linewidth=1.2,
        alpha=0.85
    )

    # Puntos de la trayectoria
    ax.scatter(
        x, y, z,
        color=color,
        s=10,
        alpha=0.9,
        depthshade=False
    )

# ==========================================
# 6. Barra de colores
# ==========================================

norm = mpl.colors.Normalize(vmin=1, vmax=max(n_timestamps, 1))

sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, pad=0.12, shrink=0.8)
cbar.set_label("Número de timestamp")

if n_timestamps <= 30:
    cbar.set_ticks(np.arange(1, n_timestamps + 1))
else:
    cbar.set_ticks(
        np.unique(np.linspace(1, n_timestamps, 10, dtype=int))
    )

# ==========================================
# 7. Configuración del gráfico
# ==========================================

ax.set_title(
    f"Trayectorias por timestamp\n"
    f"Sujeto: {subject} | Dispositivo: {device} | "
    f"Sesión: {session} | Movimiento: {movement}\n"
    f"Articulación: {articulacion}",
    fontsize=13
)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

plt.tight_layout()
plt.show()

print(f"Timestamps representados: {n_timestamps} de {len(df[(df['subject_id'] == subject) & (df['device_type'] == device) & (df['session'] == session) & (df['label'] == movement)])}")

### Analisis Temporal

#### Rrepresentación Temporal (Sujeto 8)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Classes = df_summary["activity_classes"]

# SELECCIÓN DE SUJETOS
subjects = list(df_summary["train_subject_distribution"].keys())[7:8]
subjects = [str(s) for s in subjects]

device_types = ["Commercial", "Custom"]

# COLORES POR EJE
axis_colors = {
    "x": "tab:blue",
    "y": "tab:orange",
    "z": "tab:green"
}

# IDENTIFICAR SENSORES
sensores = list(df_summary["channel_names"])

articulaciones = []
for s in sensores:
    base = s[:-2] if s.endswith(("_x", "_y", "_z")) else s
    if base not in articulaciones:
        articulaciones.append(base)

df = df.copy()
df["subject_id"] = df["subject_id"].astype(str)

# RECORRER MOVIMIENTOS
for movement in Classes:

    df_class = df[df["label"] == movement]

    if df_class.empty:
        continue

    # RECORRER SUJETOS
    for subject in subjects:

        df_subject = df_class[
            df_class["subject_id"] == subject
        ]

        if df_subject.empty:
            continue

        # RECORRER SENSORES
        for sensor in articulaciones:

            # RECORRER DISPOSITIVOS
            for dev_type in device_types:

                df_dev = df_subject[
                    df_subject["device_type"] == dev_type
                ]

                if df_dev.empty:
                    continue

                # IDENTIFICAR MUESTRAS TEMPORALES COMUNES
                times = sorted({
                    int(c.split("_t")[-1])
                    for c in df_dev.columns
                    if c.startswith(f"{sensor}_x_t")
                    and c.split("_t")[-1].isdigit()
                    and f"{sensor}_y_t{c.split('_t')[-1]}" in df_dev.columns
                    and f"{sensor}_z_t{c.split('_t')[-1]}" in df_dev.columns
                })

                if not times:
                    continue

                # UNA FIGURA CON TRES GRÁFICAS
                fig, axes = plt.subplots(
                    3, 1,
                    figsize=(16, 10),
                    sharex=True
                )

                fig.suptitle(
                    f"Movimiento: {movement} | "
                    f"Sujeto: {subject} | "
                    f"Sensor: {sensor} | "
                    f"Dispositivo: {dev_type}",
                    fontsize=14
                )

                # REPRESENTAR X, Y, Z POR SEPARADO
                for i, axis in enumerate(["x", "y", "z"]):

                    ax = axes[i]

                    cols = [
                        f"{sensor}_{axis}_t{t}"
                        for t in times
                    ]

                    # TODAS LAS TRAYECTORIAS
                    for _, row in df_dev.iterrows():

                        values = row[cols].to_numpy(dtype=float)

                        ax.plot(
                            times,
                            values,
                            color=axis_colors[axis],
                            alpha=0.25,
                            linewidth=0.8
                        )

                    ax.set_ylabel(f"Eje {axis.upper()}")
                    ax.grid(True, alpha=0.25)

                    ax.margins(x=0)
                    ax.set_xlim(times[0], times[-1])

                # TICKS DEL EJE X EN LAS TRES GRÁFICAS (fuera del bucle anterior)
                for ax in axes:
                    ax.set_xticks(times)
                    ax.set_xticklabels(
                        [f"t{t}" for t in times],
                        rotation=90,
                        fontsize=6
                    )
                    # Fuerza a mostrar las etiquetas aunque sharex=True
                    ax.tick_params(axis="x", labelbottom=True)
                    ax.set_xlabel("Muestras temporales")

                # AJUSTAR MÁRGENES (hspace mayor para que quepan las etiquetas)
                fig.subplots_adjust(
                    left=0.045,
                    right=0.99,
                    top=0.92,
                    bottom=0.08,
                    hspace=0.45
                )

                plt.show()


#### Serie Temporal con Distribución Gaussiana  

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

Classes = df_summary["activity_classes"]

# SELECCIÓN DE SUJETOS
subjects = list(df_summary["train_subject_distribution"].keys())[:1]
subjects = [str(s) for s in subjects]

device_types = ["Commercial", "Custom"]

axis_colors = {"x": "tab:blue", "y": "tab:orange", "z": "tab:green"}

# ============================================================
# OPCIONES
# ============================================================
COVERAGE = 0.997                              # cobertura de la banda gaussiana
K_SIGMA = norm.ppf(0.5 + COVERAGE / 2)        # ≈ 2.97

TRAJ_ALPHA = 0.25
TRAJ_LINEWIDTH = 1.2

# ============================================================
# IDENTIFICAR SENSORES
# ============================================================
sensores = list(df_summary["channel_names"])

articulaciones = []
for s in sensores:
    base = s[:-2] if s.endswith(("_x", "_y", "_z")) else s
    if base not in articulaciones:
        articulaciones.append(base)

df = df.copy()
df["subject_id"] = df["subject_id"].astype(str)


# ============================================================
# BANDA GAUSSIANA
# ============================================================
def gauss_center_spread(data):
    """data: (n_trayectorias, n_t). Centro = media, dispersión = desviación típica."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        center = np.nanmean(data, axis=0)
        spread = np.nanstd(data, axis=0, ddof=1)
    return center, spread


# ============================================================
# FUNCIÓN PRINCIPAL
# ============================================================
def plot_trajectories(df, df_summary):

    for movement in Classes:

        df_class = df[df["label"] == movement]
        if df_class.empty:
            continue

        for subject in subjects:

            df_subject = df_class[df_class["subject_id"] == subject]
            if df_subject.empty:
                continue

            for sensor in articulaciones:

                for dev_type in device_types:

                    df_dev = df_subject[df_subject["device_type"] == dev_type]
                    if df_dev.empty:
                        continue

                    # MUESTRAS TEMPORALES COMUNES
                    times = sorted({
                        int(c.split("_t")[-1])
                        for c in df_dev.columns
                        if c.startswith(f"{sensor}_x_t")
                        and c.split("_t")[-1].isdigit()
                        and f"{sensor}_y_t{c.split('_t')[-1]}" in df_dev.columns
                        and f"{sensor}_z_t{c.split('_t')[-1]}" in df_dev.columns
                    })
                    if not times:
                        continue

                    times_arr = np.array(times)

                    data_axes = {
                        ax_name: df_dev[[f"{sensor}_{ax_name}_t{t}" for t in times]]
                        .to_numpy(dtype=float)
                        for ax_name in ["x", "y", "z"]
                    }
                    n_traj = df_dev.shape[0]

                    # FIGURA
                    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
                    fig.suptitle(
                        f"Movimiento: {movement} | Sujeto: {subject} | "
                        f"Sensor: {sensor} | Dispositivo: {dev_type} | "
                        f"Trayectorias: {n_traj}",
                        fontsize=14
                    )

                    for i, ax_name in enumerate(["x", "y", "z"]):

                        ax = axes[i]
                        color = axis_colors[ax_name]
                        data = data_axes[ax_name]

                        center, spread = gauss_center_spread(data)

                        # BANDA GAUSSIANA
                        ax.fill_between(
                            times_arr,
                            center - K_SIGMA * spread,
                            center + K_SIGMA * spread,
                            color=color, alpha=0.15, zorder=1,
                            label=f"Banda gaussiana {COVERAGE:.1%} (±{K_SIGMA:.2f}σ)"
                        )

                        # TRAYECTORIAS
                        for j in range(n_traj):
                            ax.plot(times_arr, data[j], color=color,
                                    alpha=TRAJ_ALPHA, linewidth=TRAJ_LINEWIDTH,
                                    zorder=2)

                        # MEDIA
                        ax.plot(times_arr, center, color="black", linewidth=1.8,
                                linestyle="--", zorder=5, label="Media")

                        ax.set_ylabel(f"Eje {ax_name.upper()}")
                        ax.grid(True, alpha=0.25)
                        ax.legend(loc="upper right", fontsize=7, ncol=2)
                        ax.margins(x=0)
                        ax.set_xlim(times[0], times[-1])

                    for ax in axes:
                        ax.set_xticks(times)
                        ax.set_xticklabels([f"t{t}" for t in times],
                                           rotation=45, fontsize=6)
                        ax.tick_params(axis="x", labelbottom=True)
                        ax.set_xlabel("Muestras temporales")

                    fig.subplots_adjust(left=0.045, right=0.99, top=0.92,
                                        bottom=0.08, hspace=0.45)
                    plt.show()


# LLAMADA
plot_trajectories(df, df_summary)

##### Filtrado

In [ ]:
import warnings
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

Classes = df_summary["activity_classes"]
device_types = ["Commercial", "Custom"]

# ============================================================
# OPCIONES
# ============================================================
HALF_WIN = 3         # ventana de 5 tiempos (t-2 ... t+2)
N_SIGMAS = 3.0       # umbral (3 estándar; 3.5 más estricto; 2.5 más sensible)
SUAVE = True         # True: recorta solo el exceso; False: sustituye por la mediana del instante
K_HOOKE = 0.85       # 1.0 = el punto queda pegado al umbral; 0.75 conserva el 25 % del exceso

# ============================================================
# IDENTIFICAR SENSORES
# ============================================================
sensores = list(df_summary["channel_names"])

articulaciones = []
for s in sensores:
    base = s[:-2] if s.endswith(("_x", "_y", "_z")) else s
    if base not in articulaciones:
        articulaciones.append(base)


# ============================================================
# FILTRO POR EJE
# ============================================================
def correct_axis(data):
    """data: (n_traj, n_t) de un eje. Devuelve (data_corregido, n_puntos_corregidos).

    1) centro = mediana de todas las trayectorias en cada instante t
    2) sigma robusta = MAD de los desplazamientos en la ventana t-HALF_WIN..t+HALF_WIN
    3) si |desplazamiento| > N_SIGMAS * sigma, se corrige solo ese punto, en ese eje
    """
    half = HALF_WIN
    n_traj, n_t = data.shape

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)

        med_t = np.nanmedian(data, axis=0)                           # (n_t,)
        dev = data - med_t[None, :]                                  # desplazamiento

        padded = np.pad(dev, ((0, 0), (half, half)), constant_values=np.nan)
        win = sliding_window_view(padded, 2 * half + 1, axis=1)      # (n_traj, n_t, w)
        pool = win.transpose(1, 0, 2).reshape(n_t, -1)               # (n_t, n_traj*w)
        pool_med = np.nanmedian(pool, axis=1)
        sigma = 1.4826 * np.nanmedian(np.abs(pool - pool_med[:, None]), axis=1)

    valid = (np.isfinite(sigma) & (sigma > 0))[None, :] & np.isfinite(data)
    exceso = np.abs(dev) - N_SIGMAS * sigma[None, :]
    apply = valid & (exceso > 0)

    if SUAVE:
        corr = data - np.sign(dev) * K_HOOKE * np.where(apply, exceso, 0.0)
    else:
        corr = np.broadcast_to(med_t[None, :], data.shape)

    return np.where(apply, corr, data), int(apply.sum())


# ============================================================
# FUNCIÓN DE FILTRO
# ============================================================
def filtrar_outliers(df, df_summary, subjects=None):
    """Aplica el filtro a todas las combinaciones (movimiento, sujeto, dispositivo, sensor, eje).
    Devuelve (df_corr, df_log). df no se modifica."""

    if subjects is None:
        subjects = df["subject_id"].astype(str).unique().tolist()   # todos los sujetos
    subjects = [str(s) for s in subjects]

    df_corr = df.copy()
    subj_col = df_corr["subject_id"].astype(str)
    log = []

    for movement in Classes:
        for subject in subjects:
            for dev_type in device_types:

                idx = df_corr.index[
                    (df_corr["label"] == movement)
                    & (subj_col == subject)
                    & (df_corr["device_type"] == dev_type)
                ]
                if len(idx) == 0:
                    continue

                for sensor in articulaciones:
                    times = sorted({
                        int(c.split("_t")[-1])
                        for c in df_corr.columns
                        if c.startswith(f"{sensor}_x_t") and c.split("_t")[-1].isdigit()
                    })
                    if not times:
                        continue

                    for ax_name in ["x", "y", "z"]:
                        cols = [f"{sensor}_{ax_name}_t{t}" for t in times]
                        cols = [c for c in cols if c in df_corr.columns]
                        if not cols:
                            continue

                        data = df_corr.loc[idx, cols].to_numpy(dtype=float)
                        corr, n_pts = correct_axis(data)

                        if n_pts > 0:
                            df_corr.loc[idx, cols] = corr
                            log.append({
                                "movimiento": movement, "sujeto": subject,
                                "sensor": sensor, "dispositivo": dev_type,
                                "eje": ax_name, "puntos_corregidos": n_pts,
                            })

    return df_corr, pd.DataFrame(log)


# ============================================================
# USO Y RESUMEN
# ============================================================
df_corr, df_log = filtrar_outliers(df, df_summary)

cols_t = [c for c in df.columns if "_t" in c and pd.api.types.is_numeric_dtype(df[c])]
a = df[cols_t].to_numpy(dtype=float)
b = df_corr[cols_t].to_numpy(dtype=float)
cambiado = (a != b) & ~(np.isnan(a) & np.isnan(b))

print(f"Puntos corregidos: {int(cambiado.sum())} de {int(np.isfinite(a).sum())} "
      f"({100 * cambiado.sum() / np.isfinite(a).sum():.3f} %)")
print(f"Filas con alguna corrección: {int(cambiado.any(axis=1).sum())} de {len(df)}")

if not df_log.empty:
    print("\nPor sujeto:")
    print(df_log.groupby("sujeto")["puntos_corregidos"].sum().sort_values(ascending=False).to_string())
    print("\nPor sensor:")
    print(df_log.groupby("sensor")["puntos_corregidos"].sum().sort_values(ascending=False).to_string())
    print("\nPor eje:")
    print(df_log.groupby("eje")["puntos_corregidos"].sum().to_string())

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

Classes = df_summary["activity_classes"]

# SELECCIÓN DE SUJETOS
subjects = list(df_summary["train_subject_distribution"].keys())[:1]
subjects = [str(s) for s in subjects]

device_types = ["Commercial", "Custom"]

axis_colors = {"x": "tab:blue", "y": "tab:orange", "z": "tab:green"}

# ============================================================
# OPCIONES
# ============================================================
COVERAGE = 0.997                              # cobertura de la banda gaussiana
K_SIGMA = norm.ppf(0.5 + COVERAGE / 2)        # ≈ 2.97
TRAJ_ALPHA = 0.25
TRAJ_LINEWIDTH = 1.2

# ============================================================
# IDENTIFICAR SENSORES
# ============================================================
sensores = list(df_summary["channel_names"])

articulaciones = []
for s in sensores:
    base = s[:-2] if s.endswith(("_x", "_y", "_z")) else s
    if base not in articulaciones:
        articulaciones.append(base)

df = df.copy()
df["subject_id"] = df["subject_id"].astype(str)


# ============================================================
# BANDA GAUSSIANA
# ============================================================
def gauss_center_spread(data):
    """data: (n_trayectorias, n_t). Centro = media, dispersión = desviación típica."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        center = np.nanmean(data, axis=0)
        spread = np.nanstd(data, axis=0, ddof=1)
    return center, spread


# ============================================================
# FUNCIÓN PRINCIPAL
# ============================================================
def plot_trajectories(df, df_summary):

    for movement in Classes:

        df_class = df[df["label"] == movement]
        if df_class.empty:
            continue

        for subject in subjects:

            df_subject = df_class[df_class["subject_id"] == subject]
            if df_subject.empty:
                continue

            for sensor in articulaciones:

                for dev_type in device_types:

                    df_dev = df_subject[df_subject["device_type"] == dev_type]
                    if df_dev.empty:
                        continue

                    # MUESTRAS TEMPORALES COMUNES
                    times = sorted({
                        int(c.split("_t")[-1])
                        for c in df_dev.columns
                        if c.startswith(f"{sensor}_x_t")
                        and c.split("_t")[-1].isdigit()
                        and f"{sensor}_y_t{c.split('_t')[-1]}" in df_dev.columns
                        and f"{sensor}_z_t{c.split('_t')[-1]}" in df_dev.columns
                    })
                    if not times:
                        continue

                    times_arr = np.array(times)

                    data_axes = {
                        ax_name: df_dev[[f"{sensor}_{ax_name}_t{t}" for t in times]]
                        .to_numpy(dtype=float)
                        for ax_name in ["x", "y", "z"]
                    }
                    n_traj = df_dev.shape[0]

                    # FIGURA
                    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
                    fig.suptitle(
                        f"Movimiento: {movement} | Sujeto: {subject} | "
                        f"Sensor: {sensor} | Dispositivo: {dev_type} | "
                        f"Trayectorias: {n_traj}",
                        fontsize=14
                    )

                    for i, ax_name in enumerate(["x", "y", "z"]):

                        ax = axes[i]
                        color = axis_colors[ax_name]
                        data = data_axes[ax_name]

                        center, spread = gauss_center_spread(data)

                        # BANDA GAUSSIANA
                        ax.fill_between(
                            times_arr,
                            center - K_SIGMA * spread,
                            center + K_SIGMA * spread,
                            color=color, alpha=0.15, zorder=1,
                            label=f"Banda gaussiana {COVERAGE:.1%} (±{K_SIGMA:.2f}σ)"
                        )

                        # TRAYECTORIAS
                        for j in range(n_traj):
                            ax.plot(times_arr, data[j], color=color,
                                    alpha=TRAJ_ALPHA, linewidth=TRAJ_LINEWIDTH,
                                    zorder=2)

                        # MEDIA
                        ax.plot(times_arr, center, color="black", linewidth=1.8,
                                linestyle="--", zorder=5, label="Media")

                        ax.set_ylabel(f"Eje {ax_name.upper()}")
                        ax.grid(True, alpha=0.25)
                        ax.legend(loc="upper right", fontsize=7, ncol=2)
                        ax.margins(x=0)
                        ax.set_xlim(times[0], times[-1])

                    for ax in axes:
                        ax.set_xticks(times)
                        ax.set_xticklabels([f"t{t}" for t in times],
                                           rotation=45, fontsize=6)
                        ax.tick_params(axis="x", labelbottom=True)
                        ax.set_xlabel("Muestras temporales")

                    fig.subplots_adjust(left=0.045, right=0.99, top=0.92,
                                        bottom=0.08, hspace=0.45)
                    plt.show()


# LLAMADA
plot_trajectories(df_corr, df_summary)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Classes = df_summary["activity_classes"]
subjects = list(df_summary["train_subject_distribution"].keys())
sensores = list(df_summary["channel_names"])

# Identificar articulaciones
articulaciones = []

for s in sensores:
    base = s[:-2] if s.endswith(("_x", "_y", "_z")) else s

    if base not in articulaciones:
        articulaciones.append(base)

# Colores por sujeto
cmap = plt.get_cmap("tab20", max(len(subjects), 1))

subject_colors = {
    subj: cmap(i) for i, subj in enumerate(subjects)
}

# Número de instantes temporales
n_timesteps = 200


def plot_3d_corregido(df_plot):

    for type_class in Classes:

        df_class = df_plot[df_plot["label"] == type_class]

        if df_class.empty:
            continue

        for start in range(0, len(articulaciones), 4):

            bloque = articulaciones[start:start + 4]

            fig, axes = plt.subplots(
                1, 4,
                figsize=(24, 6),
                subplot_kw={"projection": "3d"}
            )

            fig.suptitle(
                f"Movimiento: {type_class} | DF corregido | Todos los sujetos "
                f"({start // 4 + 1})",
                fontsize=14
            )

            for idx, ax in enumerate(axes):

                ax.grid(True)

                if idx >= len(bloque):
                    ax.set_visible(False)
                    continue

                sensor = bloque[idx]

                cols_x = [f"{sensor}_x_t{t}" for t in range(n_timesteps)
                          if f"{sensor}_x_t{t}" in df_class.columns]
                cols_y = [f"{sensor}_y_t{t}" for t in range(n_timesteps)
                          if f"{sensor}_y_t{t}" in df_class.columns]
                cols_z = [f"{sensor}_z_t{t}" for t in range(n_timesteps)
                          if f"{sensor}_z_t{t}" in df_class.columns]

                if not (cols_x and cols_y and cols_z):
                    continue

                for subject in subjects:

                    df_subj = df_class[
                        df_class["subject_id"].astype(str) == str(subject)
                    ]

                    if df_subj.empty:
                        continue

                    arr_x = df_subj[cols_x].to_numpy(dtype=float)
                    arr_y = df_subj[cols_y].to_numpy(dtype=float)
                    arr_z = df_subj[cols_z].to_numpy(dtype=float)

                    for i in range(len(df_subj)):
                        ax.scatter(
                            arr_x[i], arr_y[i], arr_z[i],
                            color=subject_colors[subject],
                            s=5, alpha=0.35,
                            label=str(subject) if i == 0 else None
                        )

                ax.set_title(sensor)
                ax.set_xlabel("X")
                ax.set_ylabel("Y")
                ax.set_zlabel("Z")

            # Leyenda
            handles_dict = {}

            for ax in axes:
                if not ax.get_visible():
                    continue

                handles, labels = ax.get_legend_handles_labels()

                for handle, label in zip(handles, labels):
                    if label not in handles_dict:
                        handles_dict[label] = handle

            labels_ordered = [str(s) for s in subjects if str(s) in handles_dict]
            handles_ordered = [handles_dict[l] for l in labels_ordered]

            if handles_ordered:
                fig.legend(
                    handles_ordered,
                    labels_ordered,
                    loc="upper right",
                    ncol=2,
                    title="Sujetos"
                )

            plt.tight_layout(rect=[0, 0, 0.92, 0.94])
            plt.show()


# LLAMADA
plot_3d_corregido(df_corr)